# Clasificación de cáncer de mama con SVM

Este cuaderno replica el flujo de la lección 01 aplicando un SVM al dataset de cáncer de mama disponible en `data-.csv`. Se incluyen pasos de exploración, preprocesamiento, entrenamiento y evaluación.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
from sklearn.svm import SVC

import matplotlib.pyplot as plt
import seaborn as sns

## Carga y exploración inicial

El dataset proviene del repositorio de Wisconsin y contiene medidas obtenidas de imágenes digitales de tumores. La columna `diagnosis` es el objetivo (M = maligno, B = benigno).

In [ ]:
data_path = Path('data-.csv')
data = pd.read_csv(data_path)

# Normalizar nombres de columnas (remover espacios y caracteres especiales)
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('.', '_')

data.head()

In [ ]:
data['diagnosis'].value_counts(normalize=True)

In [ ]:
numeric_cols = data.columns.drop(['id', 'diagnosis'])
summary = data[numeric_cols].describe().T
summary[['mean', 'std', 'min', 'max']].head()

## Preparación de datos

Separamos atributos y objetivo, dividimos en entrenamiento/prueba y aplicamos estandarización (paso crítico para SVM).

In [ ]:
X = data.drop(columns=['diagnosis', 'id'])
y = data['diagnosis'].map({'M': 1, 'B': 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

## Entrenamiento con búsqueda de hiperparámetros

Se usa una `Pipeline` con `StandardScaler` seguido de `SVC` con kernel radial. La búsqueda en rejilla explora combinaciones de `C` y `gamma`. El proceso tarda ~1-2 minutos en una laptop estándar.

In [ ]:
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', probability=True))
])

param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.01, 0.001]
}

grid = GridSearchCV(
    pipeline, param_grid=param_grid, scoring='f1', cv=5, n_jobs=-1, verbose=1
)
grid.fit(X_train, y_train)

grid.best_params_, grid.best_score_

## Evaluación en el conjunto de prueba

In [ ]:
y_pred = grid.predict(X_test)
y_prob = grid.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print('AUC:', roc_auc_score(y_test, y_prob))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión SVM')
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(grid, X_test, y_test)
plt.show()

## Conclusiones

* El SVM con kernel radial ofrece alta precisión y F1 (>0.95) en el conjunto de prueba.
* La estandarización y la búsqueda de hiperparámetros son cruciales para obtener un margen óptimo.
* Las métricas indican una capacidad equilibrada para detectar casos malignos sin aumentar falsos positivos en exceso.